In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/gayatrijoshi663@gmail.com/regis-healthcare/1_setup/utility

In [0]:
# %run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","admissions","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "admission_id",
    F.trim(F.col("admission_id"))
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "admission_date",
    F.trim(F.col("admission_date"))
).withColumn(
    "admission_type",
    F.trim(F.col("admission_type"))
).withColumn(
    "referrinAg_doctor",
    F.trim(F.col("referring_doctor"))
).withColumn(
    "ward",
    F.trim(F.col("ward"))
).withColumn(
    "room_number",
    F.trim(F.col("room_number"))
).withColumn(
    "status",
    F.trim(F.col("status"))
).withColumn(
    "notes",
    F.trim(F.col("notes"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# 'admission_id',

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("admission_id").rlike("^ADM"))

df_silver = df_silver.withColumn(
    "admission_id",
    when(
        (col("admission_id").isNull()) | (~col("admission_id").rlike("^ADM")),
        "0"
    ).otherwise(col("admission_id"))
)

display(df_filt)
display(df_silver)

In [0]:
#  'resident_id',

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("resident_id").rlike("^RES"))

df_silver = df_silver.withColumn(
    "resident_id",
    when(
        (col("resident_id").isNull()) | (~col("resident_id").rlike("^RES")),
        "0"
    ).otherwise(col("resident_id"))
)

display(df_filt)
display(df_silver)

In [0]:
#  'facility_id'

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("facility_id").rlike("^FAC"))

df_silver = df_silver.withColumn(
    "facility_id",
    when(
        (col("facility_id").isNull()) | (~col("facility_id").rlike("^FAC")),
        "0"
    ).otherwise(col("facility_id"))
)

display(df_filt)
display(df_silver)

In [0]:
#  'admission_date',
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = df_silver.withColumn(
    "admission_date",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("admission_date")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("admission_date")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = df_silver.withColumn("admission_date", F.to_date("admission_date"))
# display(df_silver)

from pyspark.sql.functions import col, when, current_date

df_silver = df_silver.withColumn(
    "admission_date",
    when(
        col("admission_date").isNull(),
        current_date()
    ).otherwise(col("admission_date"))
)

display(df_silver)

In [0]:
#  'admission_type',
from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "admission_type",
    when(col("admission_type").isNull() |
        lower(trim(col("admission_type"))).isin(invalid_values),lit("not provided")
    ).otherwise(lower(trim(col("admission_type"))))
)
# display(df_silver)
dup= df_silver.groupBy("admission_type").count()
# display(dup)


In [0]:
#  'referring_doctor'

dup= df_silver.groupBy("referring_doctor").count()
display(dup)

In [0]:
#  'ward',
from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "ward",
    when(col("ward").isNull() |
        lower(trim(col("ward"))).isin(invalid_values),lit("not provided")
    ).otherwise(lower(trim(col("ward"))))
)
# display(df_silver)
dup= df_silver.groupBy("ward").count()
display(dup)

In [0]:
#  'room_number
from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "room_number",
    when(col("room_number").isNull() |
        lower(trim(col("room_number"))).isin(invalid_values),lit("not active")
    ).otherwise(lower(trim(col("room_number"))))
)
display(df_silver)
# dup= df_silver.groupBy("room_number").count()
# display(dup)

In [0]:
#  'status',
from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "status",
    when(col("status").isNull() |
        lower(trim(col("status"))).isin(invalid_values),lit("not active")
    ).otherwise(lower(trim(col("status"))))
)
display(df_silver)
# dup= df_silver.groupBy("status").count()
# display(dup)

In [0]:
#  'notes',

from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "notes",
    when(col("notes").isNull() |
        lower(trim(col("notes"))).isin(invalid_values),lit("UNKNOWN")
    ).otherwise(lower(trim(col("notes"))))
)
display(df_silver)
# dup= df_silver.groupBy("notes").count()
# display(dup)

In [0]:
 'created_at
dup= df_silver.groupBy("created_at").count()
display(dup)

In [0]:
# 'admission_id',
#  'resident_id',
#  'facility_id',
#  'admission_date',
#  'admission_type',
#  'referring_doctor',
#  'ward',
#  'room_number',
#  'status',
#  'notes',
# # 'created_at',

silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")